# AgroMind v4: yield-grounded crop ranking + conformal prediction sets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravixalgorithm/agromind/blob/main/notebooks/AgroMind_v4_Novel_Method.ipynb)

**Research question.** Published crop recommenders report ~99% accuracy, mostly on the Kaggle crop table split at random.
How good are they on **real Government of India data**, in **states and years the model has never seen**, and can we do better?

* **Data:** DES district × season × year crop statistics (1997–2020) + NASA POWER climate + SoilGrids soil (built by `backend/scripts/build_large_dataset.py`).
* **Task:** for each district-season-year, rank the crops grown there by realised yield percentile. The "best crop" is the top-yielding one.
* **Baselines (as in the literature):** Gaussian NB, Random Forest, XGBoost, MLP classifiers, plus naive *crop prior* and *last-year persistence*.
* **Ours (YieldRank):** LightGBM LambdaMART ranker on (district-season, crop) pairs, and **split-conformal prediction sets** with a 90% coverage target, calibrated on **held-out states**.
* **Protocols:** random 5-fold, **leave-states-out** (5 folds over states), **leave-years-out** (5 contiguous year blocks).

Everything below is computed live; nothing is pasted in. The code lives in `experiments/run.py` (fetched from GitHub on Colab).

In [1]:
# Setup: works locally (repo checkout) and on Colab (downloads from GitHub)
import os, sys, subprocess, urllib.request
from pathlib import Path
RAW = "https://raw.githubusercontent.com/ravixalgorithm/agromind/main/"
root = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "experiments/run.py").exists()), None)
if root is None:  # Colab
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm", "xgboost", "pyarrow"], check=True)
    root = Path("agromind");
    for f in ["experiments/run.py", "backend/data/large/agromind_india_large.parquet", "backend/data/Crop_recommendation.csv"]:
        (root / f).parent.mkdir(parents=True, exist_ok=True)
        if not (root / f).exists():
            urllib.request.urlretrieve(RAW + f, root / f)
sys.path.insert(0, str(root / "experiments"))
import numpy as np, pandas as pd, run
from scipy.stats import wilcoxon
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("repo root:", root.resolve())

repo root: /Users/originkit/.config/baton/worktrees/agromind-334992/claude-5


## 1. The literature's setting: Kaggle crop table, random 80/20 split
This reproduces the ~99% figures (2,200 rows, 100 per crop, no location or year).

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
k = pd.read_csv(root / "backend/data/Crop_recommendation.csv")
Xtr, Xte, ytr, yte = train_test_split(k.drop(columns="label"), k.label, test_size=0.2, stratify=k.label, random_state=0)
for name, m in [("Random Forest", RandomForestClassifier(300, random_state=0)), ("Gaussian NB", GaussianNB())]:
    print(f"{name:14s} accuracy on Kaggle random split: {100*(m.fit(Xtr, ytr).predict(Xte) == yte).mean():.2f}%")

Random Forest  accuracy on Kaggle random split: 99.55%
Gaussian NB    accuracy on Kaggle random split: 99.55%


## 2. Real data: DES district-season-year crop statistics

In [3]:
df = run.load()
crops = sorted(df.crop_slug.unique())
QX, Qy, Q = run.query_table(df, crops, hist=True)
print(f"(query, crop) rows : {len(df):,}")
print(f"queries            : {df.qid.nunique():,}  (district x season x year)")
print(f"crops / states     : {len(crops)} / {df.state.nunique()}   years {df.year.min()}-{df.year.max()}")
print("candidates per query:", df.groupby('qid').size().describe()[['mean','50%','max']].round(1).to_dict())
print("soil source share  :", df.soil_source.value_counts(normalize=True, dropna=False).round(3).to_dict())
df[['state','district','year','season','crop_slug','yield','yield_pct','temperature','rainfall','lag1_pct']].head(8)

(query, crop) rows : 227,451
queries            : 38,068  (district x season x year)
crops / states     : 49 / 35   years 1997-2020
candidates per query: {'mean': 6.0, '50%': 5.0, 'max': 27.0}
soil source share  : {'state_median': 0.764, 'soilgrids_centroid': 0.216, nan: 0.021}


,state,district,year,season,crop_slug,yield,yield_pct,temperature,rainfall,lag1_pct
0,Andaman and Nicobar Islands,Andaman and Nicobar Islands,2008,Autumn,arecanut,0.7379,0.2069,28.05,2157.4,NaN
1,Andaman and Nicobar Islands,Andaman and Nicobar Islands,2008,Autumn,banana,9.6981,0.1538,28.05,2157.4,NaN
2,Andaman and Nicobar Islands,Andaman and Nicobar Islands,2008,Autumn,blackpepper,0.0408,0.0513,28.05,2157.4,NaN
3,Andaman and Nicobar Islands,Andaman and Nicobar Islands,2008,Autumn,rice,1.8646,0.5073,28.05,2157.4,NaN
4,Andaman and Nicobar Islands,Andaman and Nicobar Islands,2008,Autumn,sugarcane,19.3162,0.0609,28.05,2157.4,NaN
5,Andaman and Nicobar Islands,Andaman and Nicobar Islands,2009,Autumn,arecanut,0.7513,0.2763,28.50,1927.5,0.2069
6,Andaman and Nicobar Islands,Andaman and Nicobar Islands,2009,Autumn,banana,9.3170,0.1551,28.50,1927.5,0.1538
7,Andaman and Nicobar Islands,Andaman and Nicobar Islands,2009,Autumn,blackpepper,0.0683,0.0488,28.50,1927.5,0.0513


## 3. The method (source shown for transparency)
`fit_rank` = YieldRank (LambdaMART on yield-percentile relevance); `conformal` = LAC / APS split-conformal sets.

In [4]:
import inspect
print(inspect.getsource(run.fit_rank)); print(inspect.getsource(run.conformal))

def fit_rank(df, tr, crops, hist, objective):
    d = df.loc[tr]
    X = pair_features(d, crops, hist)
    rel = np.clip(np.floor(d.yield_pct * 10), 0, 9).astype(int)  # graded 0..9
    params = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)
    if objective == "lambdarank":
        m = lgb.LGBMRanker(objective="lambdarank", label_gain=list(range(10)), lambdarank_truncation_level=10, **params)
        m.fit(X, rel, group=d.groupby("qid", sort=False).size().values)
    else:
        m = lgb.LGBMRegressor(objective="regression", **params)
        m.fit(X, d.yield_pct)
    return lambda idx: m.predict(pair_features(df.loc[idx], crops, hist))

def conformal(cal, test):
    """cal/test: DataFrames with qid, p, best. LAC + APS sets; returns coverage, size per alpha."""
    res, per = {}, {}
    for method in ("LAC", "APS"):
        def scores(d):
          

In [5]:
MODELS = ["Persistence", "GaussianNB [6,12]", "RandomForest [8,21]", "XGBoost [11,22]", "MLP [13]", "YieldRank (ours)"]
rng = np.random.default_rng(run.SEED)
idx = lambda qs: df.index[df.qid.isin(qs)]

def evaluate(protocol):
    rows, perq, conf = [], [], []
    for f, (trq, teq, groups) in enumerate(run.folds(df, protocol)):
        a, b = next(run.GroupShuffleSplit(1, test_size=0.2, random_state=run.SEED + f).split(trq, groups=groups.loc[trq].values))
        grp_cal, rest = trq[b], trq[a]
        iid_cal = rng.choice(rest, size=len(grp_cal), replace=False)
        fit_i, te_i, gc_i, ic_i = idx(np.setdiff1d(rest, iid_cal)), idx(teq), idx(grp_cal), idx(iid_cal)
        for name in MODELS:
            score = run.MODELS[name](df, fit_i, crops, True, QX, Qy)
            s_te = score(te_i)
            pq = run.per_query(df.loc[te_i], s_te)
            s_gc, s_ic = score(gc_i), score(ic_i)
            T = run.fit_T(df.loc[ic_i, "qid"], s_ic, df.loc[ic_i, "best"].values)
            mk = lambda i, s: df.loc[i, ["qid", "best"]].assign(p=run.softmax_by_q(df.loc[i, "qid"], s, T))
            ci, per_i = run.conformal(mk(ic_i, s_ic), mk(te_i, s_te))
            cg, per_g = run.conformal(mk(gc_i, s_gc), mk(te_i, s_te))
            r = dict(fold=f, model=name.split(" [")[0], ndcg3=pq["ndcg@3"].mean(), hit1=100*pq["hit@1"].mean(), hit3=100*pq["hit@3"].mean(),
                     cov_iid=100*ci["LAC@0.9"]["coverage"], cov_heldout=100*cg["LAC@0.9"]["coverage"], set_size=cg["LAC@0.9"]["size"])
            rows.append(r)
            print(f"fold {f} | {r['model']:13s} | NDCG@3 {r['ndcg3']:.3f} | Hit@1 {r['hit1']:5.1f}% | Hit@3 {r['hit3']:5.1f}% | "
                  f"90% set coverage: iid-cal {r['cov_iid']:5.1f}%  held-out-cal {r['cov_heldout']:5.1f}%  (avg size {r['set_size']:.2f})", flush=True)
            perq.append(pq.assign(model=r["model"], fold=f).reset_index())
            conf.append(per_i.add_suffix("_iid").join(per_g.add_suffix("_grp")).assign(model=r["model"], fold=f,
                        state=lambda d: Q.state.reindex(d.index).values).reset_index(names="qid"))
    R = pd.DataFrame(rows)
    summ = R.groupby("model", sort=False)[["ndcg3", "hit1", "hit3", "cov_iid", "cov_heldout", "set_size"]].agg(["mean", "std"]).round(3)
    return R, summ, pd.concat(perq), pd.concat(conf)

def significance(P, base):
    o = P[P.model == "YieldRank (ours)"].set_index(["fold", "qid"])
    for b in base:
        x = P[P.model == b].set_index(["fold", "qid"]).reindex(o.index)
        for k in ["hit@1", "ndcg@3"]:
            d = (o[k] - x[k]).dropna()
            print(f"YieldRank vs {b:13s} {k:7s}: mean diff {d.mean():+.4f}   Wilcoxon p = {wilcoxon(d).pvalue:.2e}   (n = {len(d):,} test queries)")

## 4. Random split (optimistic, for reference)

In [6]:
R_rand, S_rand, P_rand, C_rand = evaluate("random")
S_rand

fold 0 | Persistence   | NDCG@3 0.910 | Hit@1  53.8% | Hit@3  86.7% | 90% set coverage: iid-cal  89.6%  held-out-cal  89.9%  (avg size 3.14)


fold 0 | GaussianNB    | NDCG@3 0.822 | Hit@1  33.8% | Hit@3  73.5% | 90% set coverage: iid-cal  90.4%  held-out-cal  90.0%  (avg size 4.32)


fold 0 | RandomForest  | NDCG@3 0.917 | Hit@1  57.7% | Hit@3  88.9% | 90% set coverage: iid-cal  90.5%  held-out-cal  91.1%  (avg size 2.83)


fold 0 | XGBoost       | NDCG@3 0.913 | Hit@1  57.5% | Hit@3  87.9% | 90% set coverage: iid-cal  90.1%  held-out-cal  90.6%  (avg size 2.96)


fold 0 | MLP           | NDCG@3 0.901 | Hit@1  53.4% | Hit@3  85.6% | 90% set coverage: iid-cal  90.1%  held-out-cal  90.2%  (avg size 3.18)


fold 0 | YieldRank (ours) | NDCG@3 0.938 | Hit@1  60.2% | Hit@3  90.3% | 90% set coverage: iid-cal  90.4%  held-out-cal  90.3%  (avg size 2.60)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.914 | Hit@1  56.4% | Hit@3  87.5% | 90% set coverage: iid-cal  89.4%  held-out-cal  89.2%  (avg size 2.86)


fold 4 | MLP           | NDCG@3 0.899 | Hit@1  52.8% | Hit@3  85.4% | 90% set coverage: iid-cal  89.6%  held-out-cal  89.3%  (avg size 3.08)


fold 4 | YieldRank (ours) | NDCG@3 0.939 | Hit@1  60.1% | Hit@3  90.1% | 90% set coverage: iid-cal  89.4%  held-out-cal  89.5%  (avg size 2.55)


ndcg3           hit1           hit3        cov_iid        cov_heldout        set_size       
                   mean    std    mean    std    mean    std    mean    std        mean    std     mean    std
model                                                                                                         
Persistence       0.912  0.001  54.460  0.541  86.634  0.358  89.858  0.569      90.047  0.859    3.156  0.051
GaussianNB        0.821  0.002  33.170  0.462  73.253  0.694  90.086  0.387      90.049  0.192    4.358  0.028
RandomForest      0.917  0.001  57.607  0.771  88.815  0.465  90.086  0.422      90.165  0.699    2.769  0.050
XGBoost           0.914  0.001  57.392  0.776  87.903  0.352  89.863  0.342      90.097  0.759    2.906  0.063
MLP               0.901  0.001  53.728  0.769  85.613  0.541  90.047  0.519      90.091  1.024    3.159  0.084
YieldRank (ours)  0.938  0.001  59.848  0.539  90.165  0.354  89.792  0.540      90.199  0.519    2.616  0.047

## 5. Leave-states-out: train on some states, test on states never seen

In [7]:
R_lso, S_lso, P_lso, C_lso = evaluate("leave-states-out")
S_lso

fold 0 | Persistence   | NDCG@3 0.912 | Hit@1  51.4% | Hit@3  82.3% | 90% set coverage: iid-cal  88.8%  held-out-cal  83.7%  (avg size 2.85)


fold 0 | GaussianNB    | NDCG@3 0.774 | Hit@1  27.5% | Hit@3  58.7% | 90% set coverage: iid-cal  81.9%  held-out-cal  79.9%  (avg size 4.39)


fold 0 | RandomForest  | NDCG@3 0.841 | Hit@1  32.8% | Hit@3  68.3% | 90% set coverage: iid-cal  72.1%  held-out-cal  86.2%  (avg size 5.16)


fold 0 | XGBoost       | NDCG@3 0.844 | Hit@1  36.5% | Hit@3  69.1% | 90% set coverage: iid-cal  74.6%  held-out-cal  85.0%  (avg size 4.87)


fold 0 | MLP           | NDCG@3 0.833 | Hit@1  34.4% | Hit@3  68.7% | 90% set coverage: iid-cal  68.7%  held-out-cal  85.3%  (avg size 4.69)


fold 0 | YieldRank (ours) | NDCG@3 0.920 | Hit@1  51.7% | Hit@3  84.3% | 90% set coverage: iid-cal  88.3%  held-out-cal  85.5%  (avg size 2.79)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.829 | Hit@1  36.8% | Hit@3  72.0% | 90% set coverage: iid-cal  77.9%  held-out-cal  88.3%  (avg size 4.57)


fold 4 | MLP           | NDCG@3 0.819 | Hit@1  36.7% | Hit@3  71.3% | 90% set coverage: iid-cal  77.2%  held-out-cal  87.0%  (avg size 4.27)


fold 4 | YieldRank (ours) | NDCG@3 0.922 | Hit@1  56.0% | Hit@3  89.0% | 90% set coverage: iid-cal  91.0%  held-out-cal  88.7%  (avg size 2.59)


ndcg3           hit1           hit3        cov_iid        cov_heldout        set_size       
                   mean    std    mean    std    mean    std    mean    std        mean    std     mean    std
model                                                                                                         
Persistence       0.906  0.013  53.660  2.157  85.799  2.568  89.027  1.764      89.060  4.067    3.147  0.252
GaussianNB        0.782  0.028  27.086  2.639  65.706  6.154  88.147  7.317      88.240  7.789    4.459  0.342
RandomForest      0.819  0.029  34.292  2.480  69.693  2.982  73.581  5.046      88.768  2.737    4.971  0.315
XGBoost           0.842  0.020  38.800  2.799  73.652  4.219  77.947  3.972      88.393  3.499    4.566  0.206
MLP               0.829  0.032  37.615  4.494  72.384  4.444  74.307  5.828      87.906  3.192    4.449  0.233
YieldRank (ours)  0.917  0.009  54.191  2.370  87.218  2.295  90.071  1.704      89.822  3.847    2.956  0.313

In [8]:
significance(P_lso, ["Persistence", "GaussianNB", "RandomForest", "XGBoost", "MLP"])

YieldRank vs Persistence   hit@1  : mean diff +0.0053   Wilcoxon p = 2.65e-02   (n = 38,068 test queries)
YieldRank vs Persistence   ndcg@3 : mean diff +0.0108   Wilcoxon p = 8.52e-66   (n = 38,068 test queries)
YieldRank vs GaussianNB    hit@1  : mean diff +0.2711   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs GaussianNB    ndcg@3 : mean diff +0.1351   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)


YieldRank vs RandomForest  hit@1  : mean diff +0.1990   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs RandomForest  ndcg@3 : mean diff +0.0978   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs XGBoost       hit@1  : mean diff +0.1539   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs XGBoost       ndcg@3 : mean diff +0.0751   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs MLP           hit@1  : mean diff +0.1658   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)


YieldRank vs MLP           ndcg@3 : mean diff +0.0875   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)


### Conformal coverage per held-out state (YieldRank, 90% target, LAC)
`cov_iid` = calibrated on random queries from training states; `cov_heldout` = calibrated on held-out training states.

In [9]:
c = C_lso[C_lso.model == "YieldRank (ours)"]
per_state = (c.groupby("state").agg(test_queries=("qid", "size"), cov_iid=("cov_LAC_iid", "mean"), cov_heldout=("cov_LAC_grp", "mean"),
                                     avg_set_size=("size_LAC_grp", "mean")))
per_state[["cov_iid", "cov_heldout"]] *= 100
per_state = per_state[per_state.test_queries >= 30].sort_values("test_queries", ascending=False).round(2)
print(f"states below 85% coverage: iid-cal {(per_state.cov_iid < 85).sum()} / {len(per_state)},  held-out-cal {(per_state.cov_heldout < 85).sum()} / {len(per_state)}")
per_state

states below 85% coverage: iid-cal 3 / 32,  held-out-cal 3 / 32


,test_queries,cov_iid,cov_heldout,avg_set_size
state,,,,
Uttar Pradesh,4839,90.23,87.54,2.88
Madhya Pradesh,3289,91.55,91.58,3.64
Bihar,3279,91.77,95.33,3.07
Karnataka,2405,86.74,86.78,3.04
Gujarat,2177,89.66,86.82,2.61
Rajasthan,2126,89.70,87.06,2.87
Maharashtra,2072,86.78,86.87,3.15
Odisha,2013,91.36,95.58,3.81
West Bengal,1820,91.37,91.37,2.62


## 6. Leave-years-out: train on some years, test on unseen year blocks

In [10]:
R_lyo, S_lyo, P_lyo, C_lyo = evaluate("leave-years-out")
S_lyo

fold 0 | Persistence   | NDCG@3 0.912 | Hit@1  54.7% | Hit@3  86.8% | 90% set coverage: iid-cal  89.2%  held-out-cal  89.2%  (avg size 3.06)


fold 0 | GaussianNB    | NDCG@3 0.812 | Hit@1  32.1% | Hit@3  72.3% | 90% set coverage: iid-cal  89.5%  held-out-cal  90.9%  (avg size 4.51)


fold 0 | RandomForest  | NDCG@3 0.900 | Hit@1  51.7% | Hit@3  84.9% | 90% set coverage: iid-cal  85.9%  held-out-cal  88.8%  (avg size 3.14)


fold 0 | XGBoost       | NDCG@3 0.900 | Hit@1  52.6% | Hit@3  84.6% | 90% set coverage: iid-cal  86.7%  held-out-cal  89.5%  (avg size 3.31)


fold 0 | MLP           | NDCG@3 0.894 | Hit@1  50.8% | Hit@3  83.9% | 90% set coverage: iid-cal  87.5%  held-out-cal  89.6%  (avg size 3.40)


fold 0 | YieldRank (ours) | NDCG@3 0.928 | Hit@1  57.2% | Hit@3  88.1% | 90% set coverage: iid-cal  86.8%  held-out-cal  87.5%  (avg size 2.54)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.904 | Hit@1  53.7% | Hit@3  85.7% | 90% set coverage: iid-cal  87.5%  held-out-cal  92.4%  (avg size 3.61)


fold 4 | MLP           | NDCG@3 0.893 | Hit@1  52.5% | Hit@3  84.9% | 90% set coverage: iid-cal  88.8%  held-out-cal  91.5%  (avg size 3.47)


fold 4 | YieldRank (ours) | NDCG@3 0.932 | Hit@1  58.1% | Hit@3  89.5% | 90% set coverage: iid-cal  88.9%  held-out-cal  90.0%  (avg size 2.70)


ndcg3           hit1           hit3        cov_iid        cov_heldout        set_size       
                   mean    std    mean    std    mean    std    mean    std        mean    std     mean    std
model                                                                                                         
Persistence       0.911  0.015  54.431  3.542  86.497  2.349  89.863  1.137      89.720  1.390    3.137  0.102
GaussianNB        0.817  0.005  32.855  1.091  72.275  1.066  89.575  0.832      89.922  0.789    4.382  0.077
RandomForest      0.903  0.010  52.841  1.695  85.758  1.311  87.071  1.920      90.468  1.920    3.255  0.177
XGBoost           0.901  0.013  52.610  2.132  85.067  1.700  87.214  1.329      89.860  2.030    3.330  0.188
MLP               0.891  0.013  50.901  2.545  83.972  1.487  88.266  1.158      90.091  1.355    3.411  0.086
YieldRank (ours)  0.930  0.014  57.636  2.607  88.894  1.794  88.837  1.300      90.145  1.812    2.746  0.160

In [11]:
significance(P_lyo, ["Persistence", "GaussianNB", "RandomForest", "XGBoost", "MLP"])

YieldRank vs Persistence   hit@1  : mean diff +0.0334   Wilcoxon p = 8.36e-44   (n = 38,068 test queries)
YieldRank vs Persistence   ndcg@3 : mean diff +0.0188   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)


YieldRank vs GaussianNB    hit@1  : mean diff +0.2477   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs GaussianNB    ndcg@3 : mean diff +0.1128   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs RandomForest  hit@1  : mean diff +0.0486   Wilcoxon p = 1.70e-86   (n = 38,068 test queries)
YieldRank vs RandomForest  ndcg@3 : mean diff +0.0266   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs XGBoost       hit@1  : mean diff +0.0507   Wilcoxon p = 1.49e-93   (n = 38,068 test queries)
YieldRank vs XGBoost       ndcg@3 : mean diff +0.0291   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)
YieldRank vs MLP           hit@1  : mean diff +0.0679   Wilcoxon p = 2.09e-156   (n = 38,068 test queries)
YieldRank vs MLP           ndcg@3 : mean diff +0.0382   Wilcoxon p = 0.00e+00   (n = 38,068 test queries)


## 7. Summary: Hit@1 (%) and NDCG@3 by protocol

In [12]:
summary = pd.concat({"random": S_rand, "leave-states-out": S_lso, "leave-years-out": S_lyo}, axis=0)
summary[[("hit1", "mean"), ("hit1", "std"), ("ndcg3", "mean"), ("cov_heldout", "mean"), ("set_size", "mean")]]

hit1         ndcg3 cov_heldout set_size
                                     mean    std   mean        mean     mean
                 model                                                      
random           Persistence       54.460  0.541  0.912      90.047    3.156
                 GaussianNB        33.170  0.462  0.821      90.049    4.358
                 RandomForest      57.607  0.771  0.917      90.165    2.769
                 XGBoost           57.392  0.776  0.914      90.097    2.906
                 MLP               53.728  0.769  0.901      90.091    3.159
                 YieldRank (ours)  59.848  0.539  0.938      90.199    2.616
leave-states-out Persistence       53.660  2.157  0.906      89.060    3.147
                 GaussianNB        27.086  2.639  0.782      88.240    4.459
                 RandomForest      34.292  2.480  0.819      88.768    4.971
                 XGBoost           38.800  2.799  0.842      88.393    4.566
                 MLP               37.615  4.494  0.829      87.906    4.449
                 YieldRank (ours)  54.191  2.370  0.917      89.822    2.956
leave-years-out  Persistence       54.431  3.542  0.911      89.720    3.137
                 GaussianNB        32.855  1.091  0.817      89.922    4.382
                 RandomForest      52.841  1.695  0.903      90.468    3.255
                 XGBoost           52.610  2.132  0.901      89.860    3.330
                 MLP               50.901  2.545  0.891      90.091    3.411
                 YieldRank (ours)  57.636  2.607  0.930      90.145    2.746

## Reading the numbers
* The ~99% of Section 1 is a property of the Kaggle table, not of crop choice. On real data, **no** model's top-1 pick matches the best-yielding crop anywhere near that often.
* Compare each baseline's Hit@1 between the random and leave-states-out tables: the drop is the over-optimism of random splits.
* *Persistence* (repeat last year's yield ranking) is a strong, rarely reported baseline. The gap between it and YieldRank is the honest size of the learned gain.
* Conformal coverage with i.i.d. calibration is a guarantee about *seen* states. Per-state coverage shows how it behaves on unseen ones.

The paper (`paper/main.pdf`) reports the same experiment from `experiments/run.py`.